# Round 1 exploratory analysis

- **ASH_COATED_OSMIUM**: random-walk-like, look for hidden periodicity via autocorrelation.
- **INTARIAN_PEPPER_ROOT**: near-linear drift, try a rolling OLS forecaster for next-tick bid/ask wall.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import prosperity4bt

DATA_DIR = Path(prosperity4bt.__file__).parent / "resources" / "round1"
DAYS = [-2, -1, 0]
OSM = "ASH_COATED_OSMIUM"
PPR = "INTARIAN_PEPPER_ROOT"

BID_COLS = ["bid_price_1", "bid_price_2", "bid_price_3"]
ASK_COLS = ["ask_price_1", "ask_price_2", "ask_price_3"]

def load_day(day: int) -> pd.DataFrame:
    path = DATA_DIR / f"prices_round_1_day_{day}.csv"
    df = pd.read_csv(path, sep=";")
    df["wall_bid"] = df[BID_COLS].min(axis=1, skipna=True)
    df["wall_ask"] = df[ASK_COLS].max(axis=1, skipna=True)
    df["wall_mid"] = (df["wall_bid"] + df["wall_ask"]) / 2
    return df[["day", "timestamp", "product", "wall_bid", "wall_ask", "wall_mid", "mid_price"]]

def load_all() -> pd.DataFrame:
    parts = [load_day(d) for d in DAYS]
    df = pd.concat(parts, ignore_index=True)
    ts_span = df["timestamp"].max() + 100
    df["t"] = (df["day"] - df["day"].min()) * ts_span + df["timestamp"]
    return df.sort_values(["product", "t"]).reset_index(drop=True)

df = load_all()
print(df.shape, df["product"].unique())
df.head()

In [ ]:
# Quick visual
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, prod in zip(axes, [OSM, PPR]):
    s = df[df["product"] == prod]
    ax.plot(s["t"], s["wall_mid"], lw=0.5)
    ax.set_title(f"{prod} wall_mid")
    for d in sorted(df["day"].unique())[1:]:
        t0 = df[df["day"] == d]["t"].min()
        ax.axvline(t0, color="k", lw=0.3, ls="--")
plt.tight_layout()
plt.show()

## Osmium autocorrelation

ACF on returns first (random-walks have ~zero returns ACF; deviations hint at mean reversion or cycles), then ACF on centered levels for long cycles.

In [ ]:
osm = df[df["product"] == OSM].set_index("t")["wall_mid"].astype(float)

def acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    x = x - x.mean()
    denom = np.dot(x, x)
    out = np.empty(max_lag + 1)
    out[0] = 1.0
    for k in range(1, max_lag + 1):
        out[k] = np.dot(x[:-k], x[k:]) / denom
    return out

MAX_LAG = 2000
ret = osm.diff().dropna().to_numpy()
acf_ret = acf(ret, MAX_LAG)
acf_lvl = acf(osm.to_numpy(), MAX_LAG)

sig = 2.0 / np.sqrt(len(ret))

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(acf_ret, lw=0.7)
axes[0].axhline(sig, color="r", ls="--", lw=0.6)
axes[0].axhline(-sig, color="r", ls="--", lw=0.6)
axes[0].set_title("Osmium ACF — returns")
axes[0].set_xlabel("lag")
axes[1].plot(acf_lvl, lw=0.7)
axes[1].set_title("Osmium ACF — levels (centered)")
axes[1].set_xlabel("lag")
plt.tight_layout()
plt.show()

top = np.argsort(-np.abs(acf_ret[1:]))[:10] + 1
print("Top-10 return-ACF lags (lag, acf):")
for k in top:
    print(f"  lag={k:5d}  acf={acf_ret[k]:+.4f}")

In [ ]:
# Disentangle lag-1 ACF: bid-ask bounce vs real mean reversion.
# wall_mid uses outermost levels (less bounce); mid_price is top-of-book (more bounce).
osm_rows = df[df["product"] == OSM].set_index("t")
for col in ("mid_price", "wall_mid"):
    r = osm_rows[col].astype(float).diff().dropna().to_numpy()
    a = acf(r, 5)
    print(f"{col:10s}  lag-1 ACF(returns) = {a[1]:+.4f}   lag-2 = {a[2]:+.4f}   lag-3 = {a[3]:+.4f}   n={len(r)}")

# If lag-1 on wall_mid is close to 0 and lag-1 on mid_price is strongly negative
# => artifact (bid-ask bounce at top-of-book). Not tradeable as pure signal.
# If lag-1 on wall_mid is still strongly negative => real short-horizon mean reversion.

In [ ]:
# Osmium long-running average
print(f"mean={osm.mean():.2f}  std={osm.std():.2f}  min={osm.min()}  max={osm.max()}  n={len(osm)}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(osm.index, osm.values, lw=0.4, label="wall_mid")
ax.axhline(osm.mean(), color="k", ls="--", lw=0.8, label=f"mean={osm.mean():.1f}")
for w in (500, 2000, 10000):
    ax.plot(osm.index, osm.rolling(w).mean().values, lw=1.0, label=f"rolling {w}")
ax.legend()
ax.set_title("Osmium wall_mid + rolling means")
plt.tight_layout()
plt.show()

## Pepper Root local-linear forecast

Rolling OLS over the last `N` observations → forecast next-tick `wall_bid` and `wall_ask`.

In [ ]:
ppr = df[df["product"] == PPR].set_index("t")[["wall_bid", "wall_ask"]].astype(float).dropna()

def rolling_linear_forecast(y: np.ndarray, N: int, horizon: int = 1) -> np.ndarray:
    """For each i in [N-1, len(y)-1], fit OLS on y[i-N+1 : i+1] and predict `horizon` steps ahead.
    Returns array of length len(y); first N-1 entries are NaN.
    """
    n = len(y)
    out = np.full(n, np.nan)
    x = np.arange(N)
    xm = x.mean()
    xc = x - xm
    denom = np.dot(xc, xc)
    for i in range(N - 1, n):
        window = y[i - N + 1 : i + 1]
        slope = np.dot(xc, window - window.mean()) / denom
        intercept = window.mean() - slope * xm
        out[i] = intercept + slope * (N - 1 + horizon)
    return out

Ns = [20, 50, 100, 200, 500]
rows = []
preds = {}
for N in Ns:
    row = {"N": N}
    for side in ("wall_bid", "wall_ask"):
        y = ppr[side].to_numpy()
        f = rolling_linear_forecast(y, N, horizon=1)
        # align: forecast at index i is for i+1; compare f[i] with y[i+1]
        pred = f[:-1]
        realized = y[1:]
        mask = ~np.isnan(pred)
        mae = np.mean(np.abs(pred[mask] - realized[mask]))
        row[f"MAE_{side}"] = mae
        preds[(N, side)] = (pred, realized, mask)
    rows.append(row)

mae_tbl = pd.DataFrame(rows).set_index("N")
# Baseline: predict next = current (naive)
for side in ("wall_bid", "wall_ask"):
    y = ppr[side].to_numpy()
    mae_tbl.loc["naive", f"MAE_{side}"] = np.mean(np.abs(y[:-1] - y[1:]))
mae_tbl

In [ ]:
# Eyeball fit: best N for wall_bid on the first 2000 ticks of day 0
best_N = int(mae_tbl.drop("naive").idxmin()["MAE_wall_bid"])
print(f"Best N for wall_bid = {best_N}")

ppr_day0 = df[(df["product"] == PPR) & (df["day"] == 0)].set_index("t")[["wall_bid", "wall_ask"]].astype(float).dropna()
y = ppr_day0["wall_bid"].to_numpy()
f = rolling_linear_forecast(y, best_N, horizon=1)

sl = slice(0, 2000)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ppr_day0.index[sl], y[sl], lw=0.6, label="wall_bid (realized)")
ax.plot(ppr_day0.index[sl], f[sl], lw=0.6, label=f"OLS(N={best_N}) forecast")
ax.legend()
ax.set_title(f"Pepper root wall_bid — day 0 first 2000 ticks — N={best_N}")
plt.tight_layout()
plt.show()